In [10]:
import pandas as pd
import numpy as np
from cmdstanpy import CmdStanModel
import scipy.stats as ss
from tqdm import tqdm
from IPython.display import clear_output

In [2]:
# 1) Define your Stan code as a Python string
stan_code = """
data {
  int<lower=0> N;
  real<lower=0> dt;
  array[N] real x;
}
parameters {
  real mu;
  real lambda_plus;
  real lambda_minus;
  real<lower=1e-9> sigma;
  real<lower=0,upper=1> alpha;
}
model {
  // Priors
  mu        ~ normal(0, 1);
  lambda_plus  ~ normal(0, 10);
  lambda_minus ~ normal(0, 10);
  sigma        ~ normal(0, 1);
  alpha    ~ beta(1,5);

  // Recursive EMA (internal only)
  vector[N] ema;
  ema[1] = x[1];
  ema[2] = x[2];
  for (i in 3:N)
    ema[i] = alpha * x[i-1] + (1 - alpha) * ema[i-2];

  // Likelihood
  for (n in 3:N) {
    real diff      = (ema[n-1] - x[n-1]) / x[n-1];
    real up_move   = fmax(diff, 0);
    real down_move = fmin(diff, 0);
    real theta        = x[n-1] + dt * (mu + lambda_plus * up_move + lambda_minus * down_move) * x[n-1];
    real sd        = fmax(1e-9, abs(x[n-1]) * sigma * sqrt(dt));
    x[n] ~ normal(theta, sd);
  }
}
"""

# 2) Write that string to a file called "model.stan" in your cwd
with open("beta_1_5.stan", "w") as f:
    f.write(stan_code)

# 3) Confirm the file exists
import os
print("Wrote to:", os.path.abspath("beta_1_5.stan"))
print("Directory now contains:", os.listdir(os.getcwd()))

Wrote to: C:\Users\smp22snm\my_phd_python_codes\beta_1_5.stan
Directory now contains: ['.ipynb_checkpoints', 'beta_1_5.stan', 'test_pystan.ipynb', 'Untitled.ipynb']


In [3]:
sm = CmdStanModel(stan_file="beta_1_5.stan")
print("Compiled executable:", sm.exe_file)

13:58:06 - cmdstanpy - INFO - compiling stan file C:\Users\smp22snm\my_phd_python_codes\beta_1_5.stan to exe file C:\Users\smp22snm\my_phd_python_codes\beta_1_5.exe
14:01:45 - cmdstanpy - INFO - compiled model executable: C:\Users\smp22snm\my_phd_python_codes\beta_1_5.exe


Compiled executable: C:\Users\smp22snm\my_phd_python_codes\beta_1_5.exe


In [4]:
def simulate_price(N,paths,dt,mu_sim,sigma_sim,lambda_plus_sim,lambda_minus_sim,alpha_sim):
    np.random.seed(seed=100)
    X_0_sim = 100
    X_sim = np.zeros((paths, N))
    EMA_sim = np.zeros((paths, N))
    DIFF_sim = np.zeros((paths, N))
    POS_DIFF_sim = np.zeros((paths, N))
    NEG_DIFF_sim = np.zeros((paths, N))

    X_sim[:, 0:2] = X_0_sim
    EMA_sim[:, 0:2] = X_0_sim
    DIFF_sim[:, 0:2] = 0
    POS_DIFF_sim[:, 0:2] = 0
    NEG_DIFF_sim[:, 0:2] = 0

    W = ss.norm.rvs(loc=0, scale=1, size=(paths, N))

    for t in range(1, N - 1):
        X_sim[:, t + 1] = X_sim[:, t] + X_sim[:, t] * np.sqrt(dt) * sigma_sim * (W[:, t]) + dt * (mu_sim + lambda_plus_sim * (POS_DIFF_sim[:, t]) + lambda_minus_sim * (NEG_DIFF_sim[:, t])) * X_sim[:, t]
        EMA_sim[:, t + 1] = alpha_sim * X_sim[:, t] + (1 - alpha_sim) * EMA_sim[:, t - 1]
        DIFF_sim[:, t + 1] = (EMA_sim[:, t + 1] - X_sim[:, t + 1]) / X_sim[:, t + 1]
        POS_DIFF_sim[:, t + 1] = np.where(DIFF_sim[:, t + 1] < 0, 0, DIFF_sim[:, t + 1])
        NEG_DIFF_sim[:, t + 1] = np.where(DIFF_sim[:, t + 1] > 0, 0, DIFF_sim[:, t + 1])

    df_data = pd.DataFrame()
    pairs = [f'sim_asset_{i}' for i in range(paths)]
    for i in range(paths):
        df_data[f'sim_asset_{i}'] = X_sim[i]
    return df_data

In [5]:
[mu_sim,sigma_sim,lambda_plus_sim,lambda_minus_sim,alpha_sim] =  [-0.0005,0.01,0,0,0.8]


N = 2500
paths = 20
dt = 1

simulate_price(N,paths,dt,mu_sim,sigma_sim,lambda_plus_sim,lambda_minus_sim,alpha_sim)

,sim_asset_0,sim_asset_1,sim_asset_2,sim_asset_3,sim_asset_4,sim_asset_5,sim_asset_6,sim_asset_7,sim_asset_8,sim_asset_9,sim_asset_10,sim_asset_11,sim_asset_12,sim_asset_13,sim_asset_14,sim_asset_15,sim_asset_16,sim_asset_17,sim_asset_18,sim_asset_19
0,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
1,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
2,100.292680,100.120992,98.805103,99.701843,100.593354,101.146855,99.525507,101.148467,100.647272,101.926999,100.113474,101.075984,100.229795,98.272743,99.475657,99.950182,99.772873,98.257723,100.728403,100.608614
3,101.398945,102.850616,100.639515,99.434223,99.477468,100.648449,99.529413,100.273416,98.863755,101.868209,100.906976,101.231374,98.648718,98.197797,99.547572,101.166910,101.064892,97.504429,99.662714,101.641759
4,101.092278,101.544332,100.092525,101.629615,100.075962,98.852135,99.410175,99.242323,98.338289,102.036806,101.057171,102.097890,100.219412,98.361319,99.432877,99.562229,101.734543,98.055943,99.931268,102.161552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,31.382049,32.073155,12.241184,38.239241,35.330551,35.984188,19.256411,15.782218,40.446509,36.513288,16.428399,20.272168,22.976925,24.221630,27.931188,24.441220,34.940126,47.078467,27.284110,18.396951
2496,31.428412,32.038063,12.211396,38.851618,34.899516,36.663929,19.346378,15.735662,39.958043,36.748720,16.651306,20.038845,22.746485,24.490612,27.893586,24.310476,35.136023,46.279247,27.229533,18.547220
2497,31.379287,32.381031,12.373493,38.872318,34.768922,36.559197,19.494594,15.645654,40.356443,36.800636,16.690928,20.249771,22.535780,24.496129,28.547603,24.463197,35.303935,46.421614,26.889533,18.358790
2498,31.369173,32.385390,12.239630,38.678345,34.669979,36.616293,19.413328,15.411187,40.690492,36.032730,16.395967,20.283321,22.303265,24.011287,28.491388,24.439288,35.132485,46.238493,26.459122,18.523160


In [11]:
np.random.seed(seed=42)

list_models = [
 'beta_1_5.stan'
]

cases = [
    [-0.0005,0.01,0,0,0.8]
]

N = 2500
paths = 1
dt = 1
pairs = [f'sim_asset_{i}' for i in range(paths)]

for case in cases:
    df_data = simulate_price(N,paths,dt,case[0],case[1],case[2],case[3],case[4])

    for name in tqdm(list_models):
        sm = CmdStanModel(stan_file=name)
        sigma_df = pd.DataFrame()
        lambda_plus_df = pd.DataFrame()
        lambda_minus_df = pd.DataFrame()
        mu_df = pd.DataFrame()
        alpha_df = pd.DataFrame()

        for asset in tqdm(pairs):
            y = df_data[asset].dropna().tolist()
            data = {"N": len(y), "dt": dt, "x": y}

            fit = sm.sample(
                data=data,
                chains=2,
                parallel_chains=2,
                iter_warmup=250,
                iter_sampling=250
            )

            sigma_df       [asset] = fit.stan_variable("sigma")
            lambda_plus_df [asset] = fit.stan_variable("lambda_plus")
            lambda_minus_df[asset] = fit.stan_variable("lambda_minus")
            mu_df       [asset] = fit.stan_variable("mu")
            alpha_df   [asset] = fit.stan_variable("alpha")
            clear_output()
        
        mu_str = str(case[0]).replace('.', 'p')
        sigma_str = str(case[1]).replace('.', 'p')
        lambdap_str = str(case[2]).replace('.', 'p')
        lambdam_str = str(case[3]).replace('.', 'p')
        alpha_str = str(case[4]).replace('.', 'p')

        save_name = os.path.splitext(name)[0]
        lambda_plus_df.to_csv(f'{N}_points_{mu_str}_{sigma_str}_{lambdap_str}_{lambdam_str}_{alpha_str}_{save_name}_lambda_plus.csv')
        lambda_minus_df.to_csv(f'{N}_points_{mu_str}_{sigma_str}_{lambdap_str}_{lambdam_str}_{alpha_str}_{save_name}_lambda_minus.csv')
        mu_df.to_csv(f'{N}_points_{mu_str}_{sigma_str}_{lambdap_str}_{lambdam_str}_{alpha_str}_{save_name}_mu.csv')
        alpha_df.to_csv(f'{N}_points_{mu_str}_{sigma_str}_{lambdap_str}_{lambdam_str}_{alpha_str}_{save_name}_alpha.csv')
        sigma_df.to_csv(f'{N}_points_{mu_str}_{sigma_str}_{lambdap_str}_{lambdam_str}_{alpha_str}_{save_name}_sigma.csv')


100%|██████████| 1/1 [00:54<00:00, 54.78s/it]
